<a href="https://colab.research.google.com/github/amrKidwai/RFM-Customer-Segmentation/blob/main/PROJECT_2_CUSTOMER_SEGMENTATION_(RFM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Generate Dataset

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 2000

# 1-year random dates
date_range = pd.date_range(start='2023-01-01', end='2023-12-31')

df = pd.DataFrame({
    'user_id': np.random.randint(1, 500, n),
    'order_date': np.random.choice(date_range, n),
    'order_value': np.random.randint(100, 5000, n)
})

df.to_csv('rfm_data.csv', index=False)
df.head()

,user_id,order_date,order_value
0,103,2023-05-11,2760
1,436,2023-10-20,2311
2,349,2023-06-20,2009
3,271,2023-07-02,3830
4,107,2023-12-30,2146


STEP 2: RFM CALCULATION

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])

snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('user_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'user_id': 'count',
    'order_value': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
user_id,,,
1,53,8,19804
2,125,4,10397
3,157,1,1849
4,95,5,20230
5,110,7,22478


STEP 3: CREATE RFM SCORES

In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'], 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

STEP 4: COMBINE SCORE

In [ ]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

STEP 5: CONVERT TO INT

In [ ]:
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)


STEP 6: SEGMENT CUSTOMERS

In [ ]:
def segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'

    elif row['R_score'] >= 3 and row['F_score'] >= 4:
        return 'Loyal Customers'

    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return 'New Customers'

    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'

    elif row['R_score'] == 1:
        return 'Lost Customers'

    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)
rfm=rfm.reset_index()

STEP 7: ANALYSIS

In [ ]:
rfm.groupby('Segment').agg({
    'user_id': 'count',
    'Monetary': 'mean'
})
rfm.to_csv('rfm_output.csv',index=False)